# SecondCrop — training notebook

Trains a MobileNetV2 transfer-learning classifier on the Kaggle *Fruits fresh and rotten for classification* dataset.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

The source dataset is binary (fresh vs rotten), so this trains a binary model: score near 1 = fresh (Grade A, retail), score near 0 = rotten (Grade C, rescue/compost). Grade B (blemished but edible) has no training data yet — until the supermarket photo shoot happens, the backend treats mid-range scores as "needs manual review" rather than a trained Grade B, so all three routing buckets (retail / processing-review / rescue) are usable from day one.

## 1. Get the data
Paste your Kaggle API token below (Kaggle -> Settings -> API Tokens -> Generate). Only paste this into the Colab cell, never share it elsewhere -- if a token has ever been visible outside Colab, delete it on Kaggle and generate a fresh one first.

In [ ]:
!pip install -q kaggle
import os

KAGGLE_TOKEN = ""  # paste your token here, e.g. KGAT_xxxxxxxx
assert KAGGLE_TOKEN, "Paste your Kaggle API token above before running"

!mkdir -p ~/.kaggle
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(KAGGLE_TOKEN)
!chmod 600 ~/.kaggle/access_token

In [ ]:
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification -p /content/data
!unzip -q /content/data/fruits-fresh-and-rotten-for-classification.zip -d /content/data/raw
!find /content/data/raw -maxdepth 4 -type d | head -20

## 2. Build a file list + binary labels
Folder name prefix (`fresh*` / `rotten*`) determines the label — same convention as `model/prepare_dataset.py` in the repo.

In [ ]:
import glob, os
import numpy as np

RAW_DIR = "/content/data/raw"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

all_paths = glob.glob(f"{RAW_DIR}/**/*.png", recursive=True) + \
            glob.glob(f"{RAW_DIR}/**/*.jpg", recursive=True) + \
            glob.glob(f"{RAW_DIR}/**/*.jpeg", recursive=True)

def label_for(path):
    folder = os.path.basename(os.path.dirname(path)).lower()
    if folder.startswith("fresh"):
        return 1  # Grade A
    if folder.startswith("rotten"):
        return 0  # Grade C
    return None

paths, labels = [], []
for p in all_paths:
    y = label_for(p)
    if y is not None:
        paths.append(p)
        labels.append(y)

paths, labels = np.array(paths), np.array(labels)
print(f"Total images: {len(paths)}  |  fresh: {labels.sum()}  |  rotten: {len(labels) - labels.sum()}")

In [ ]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=42
)
print(f"train: {len(train_paths)}  val: {len(val_paths)}")

## 3. tf.data pipeline

In [ ]:
import tensorflow as tf

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label

def make_ds(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2048)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_paths, train_labels, training=True)
val_ds = make_ds(val_paths, val_labels, training=False)

## 4. Model — MobileNetV2 transfer learning
Base frozen, small classifier head on top. This is what makes ~15 epochs on 27k images feasible in ~20-40 min on a T4.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
)
base.trainable = False

model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
)

## 5. Evaluate

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Validation accuracy: {acc:.3f}")

## 6. Export and download
Grab `secondcrop_model.keras` afterward and drop it into `model/` in the repo — the FastAPI `/grade` endpoint will load it from there.

In [ ]:
model.save("secondcrop_model.keras")
from google.colab import files
files.download("secondcrop_model.keras")

## 7. Routing logic reference (for the backend, not run here)
```python
score = model.predict(image)[0][0]  # 0..1, higher = fresher
if score >= 0.7:
    grade, route = "A", "retail"
elif score <= 0.3:
    grade, route = "C", "rescue"
else:
    grade, route = "B", "processing_review"  # proxy until real Grade B data exists
```